```

Estrutura do ETL

1.  Imports e carregamento
2.  Remover duplicatas
3.  Remover cancelamentos (Invoice 'C')
4.  Remover ajustes contábeis (Invoice 'A')
5.  Remover StockCodes não-produto
6.  Remover Price zero e negativo
7.  Remover Customer ID nulos
8.  Remover Description nulos
9.  Corrigir tipos — Customer ID float → string
10. Tratar outliers de Quantity
11. Criar coluna TotalPrice
12. Checkpoint — validar o dataset limpo
13. Salvar em data/processed/


### Célula 1 — Imports e carregamento

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Carregar o dataset bruto
df_0910 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_1011 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')
df = pd.concat([df_0910, df_1011], ignore_index=True)

print(f"Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

Dataset carregado: 1,067,371 linhas x 8 colunas


### Célula 2 — Remover duplicatas

In [3]:
antes = len(df)

df = df.drop_duplicates()

depois = len(df)
removidas = antes - depois

print(f"Linhas antes:    {antes:,}")
print(f"Linhas depois:   {depois:,}")
print(f"Duplicatas removidas: {removidas:,} ({round(removidas/antes*100, 2)}%)")

Linhas antes:    1,067,371
Linhas depois:   1,033,036
Duplicatas removidas: 34,335 (3.22%)


### Célula 3 — Remover cancelamentos (Invoice 'C')

In [4]:
antes = len(df)

df = df[~df['Invoice'].astype(str).str.startswith('C')]

depois = len(df)
removidas = antes - depois

print(f"Linhas antes:    {antes:,}")
print(f"Linhas depois:   {depois:,}")
print(f"Cancelamentos removidos: {removidas:,} ({round(removidas/antes*100, 2)}%)")

Linhas antes:    1,033,036
Linhas depois:   1,013,932
Cancelamentos removidos: 19,104 (1.85%)


### Célula 4 — Remover ajustes contábeis (Invoice 'A')

In [5]:
antes = len(df)

df = df[~df['Invoice'].astype(str).str.startswith('A')]

depois = len(df)
removidas = antes - depois

print(f"Linhas antes:    {antes:,}")
print(f"Linhas depois:   {depois:,}")
print(f"Ajustes contábeis removidos: {removidas:,}")

Linhas antes:    1,013,932
Linhas depois:   1,013,926
Ajustes contábeis removidos: 6


### Célula 5 — Remover StockCodes não-produto

In [6]:
antes = len(df)

# Lista de StockCodes operacionais identificados no EDA
stockcodes_nao_produto = ['POST', 'DOT', 'M', 'C2', 'D', 'S', 
                          'BANK CHARGES', 'ADJUST', 'AMAZONFEE', 
                          'CRUK', 'TEST001', 'PADS', 'B']

df = df[~df['StockCode'].astype(str).isin(stockcodes_nao_produto)]

depois = len(df)
removidas = antes - depois

print(f"Linhas antes:    {antes:,}")
print(f"Linhas depois:   {depois:,}")
print(f"StockCodes não-produto removidos: {removidas:,} ({round(removidas/antes*100, 2)}%)")

Linhas antes:    1,013,926
Linhas depois:   1,009,413
StockCodes não-produto removidos: 4,513 (0.45%)


### Célula 6 — Remover Price zero e negativo

In [7]:
antes = len(df)

df = df[df['Price'] > 0]

depois = len(df)
removidas = antes - depois

print(f"Linhas antes:    {antes:,}")
print(f"Linhas depois:   {depois:,}")
print(f"Registros com Price <= 0 removidos: {removidas:,} ({round(removidas/antes*100, 2)}%)")

Linhas antes:    1,009,413
Linhas depois:   1,003,426
Registros com Price <= 0 removidos: 5,987 (0.59%)


### Célula 7 — Remover Quantity zero e negativo

In [8]:
antes = len(df)

df = df[df['Quantity'] > 0]

depois = len(df)
removidas = antes - depois

print(f"Linhas antes:    {antes:,}")
print(f"Linhas depois:   {depois:,}")
print(f"Registros com Quantity <= 0 removidos: {removidas:,} ({round(removidas/antes*100, 2)}%)")

Linhas antes:    1,003,426
Linhas depois:   1,003,426
Registros com Quantity <= 0 removidos: 0 (0.0%)


### Célula 8 — Remover Customer ID nulos

In [9]:
antes = len(df)

df = df[df['Customer ID'].notna()]

depois = len(df)
removidas = antes - depois

print(f"Linhas antes:    {antes:,}")
print(f"Linhas depois:   {depois:,}")
print(f"Registros sem Customer ID removidos: {removidas:,} ({round(removidas/antes*100, 2)}%)")

Linhas antes:    1,003,426
Linhas depois:   776,583
Registros sem Customer ID removidos: 226,843 (22.61%)


### Célula 9 — Remover Description nulos

In [10]:
antes = len(df)

df = df[df['Description'].notna()]

depois = len(df)
removidas = antes - depois

print(f"Linhas antes:    {antes:,}")
print(f"Linhas depois:   {depois:,}")
print(f"Registros sem Description removidos: {removidas:,} ({round(removidas/antes*100, 2)}%)")

Linhas antes:    776,583
Linhas depois:   776,583
Registros sem Description removidos: 0 (0.0%)


### Célula 10 — Corrigir tipos — Customer ID float → string

In [11]:
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)

print("Tipo antes da conversão: float64")
print(f"Tipo depois da conversão: {df['Customer ID'].dtype}")
print(f"\nExemplo de valores:")
print(df['Customer ID'].head())

Tipo antes da conversão: float64
Tipo depois da conversão: str

Exemplo de valores:
0    13085
1    13085
2    13085
3    13085
4    13085
Name: Customer ID, dtype: str


### Célula 11 — Tratar outliers de Quantity

In [12]:
antes = len(df)

q99 = df['Quantity'].quantile(0.99)
df = df[df['Quantity'] <= q99]

depois = len(df)
removidas = antes - depois

print(f"Corte no percentil 99: {q99:.0f} unidades")
print(f"Linhas antes:    {antes:,}")
print(f"Linhas depois:   {depois:,}")
print(f"Outliers removidos: {removidas:,} ({round(removidas/antes*100, 2)}%)")

Corte no percentil 99: 144 unidades
Linhas antes:    776,583
Linhas depois:   770,715
Outliers removidos: 5,868 (0.76%)


### Célula 12 — Criar coluna TotalPrice

In [13]:
df['TotalPrice'] = df['Quantity'] * df['Price']

print(f"Coluna TotalPrice criada com sucesso!")
print(f"\nEstatísticas da nova coluna:")
print(f"Mínimo:  £{df['TotalPrice'].min():.2f}")
print(f"Mediana: £{df['TotalPrice'].median():.2f}")
print(f"Média:   £{df['TotalPrice'].mean():.2f}")
print(f"Máximo:  £{df['TotalPrice'].max():.2f}")
print(f"\nExemplo:")
df[['Invoice', 'Description', 'Quantity', 'Price', 'TotalPrice']].head()

Coluna TotalPrice criada com sucesso!

Estatísticas da nova coluna:
Mínimo:  £0.06
Mediana: £11.90
Média:   £18.60
Máximo:  £38970.00

Exemplo:


,Invoice,Description,Quantity,Price,TotalPrice
0,489434,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,83.4
1,489434,PINK CHERRY LIGHTS,12,6.75,81.0
2,489434,WHITE CHERRY LIGHTS,12,6.75,81.0
3,489434,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,100.8
4,489434,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,30.0


### Célula 13 — Checkpoint final — validar o dataset limpo

In [15]:
print("CHECKPOINT — DATASET APÓS LIMPEZA")
print('='*50)
print(f"\nDimensões finais: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"\nValores nulos restantes:")
print(df.isnull().sum())
print(f"\nTipos das colunas:")
print(df.dtypes)
print(f"\nEstatísticas finais:")
print(df[['Quantity', 'Price', 'TotalPrice']].describe())

# Resumo do que foi removido
total_original = 1_067_371
total_final = len(df)
removidas_total = total_original - total_final

print(f"\nRESUMO DO PIPELINE")
print('='*50)
print(f"Linhas originais:  {total_original:,}")
print(f"Linhas removidas:  {removidas_total:,} ({round(removidas_total/total_original*100,2)}%)")
print(f"Linhas mantidas:   {total_final:,} ({round(total_final/total_original*100,2)}%)")

CHECKPOINT — DATASET APÓS LIMPEZA

Dimensões finais: 770,715 linhas x 9 colunas

Valores nulos restantes:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
TotalPrice     0
dtype: int64

Tipos das colunas:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
TotalPrice            float64
dtype: object

Estatísticas finais:
            Quantity          Price     TotalPrice
count  770715.000000  770715.000000  770715.000000
mean       10.188720       2.947907      18.596191
std        16.134741       4.338838      55.722965
min         1.000000       0.040000       0.060000
25%         2.000000       1.250000       4.950000
50%         5.000000       1.950000      11.900000
75%        12.000000       3.750000      19.5

### Célula 14 — Salvar dataset processado

In [16]:
df.to_csv('../data/processed/online_retail_clean.csv', index=False)

print(f"Dataset salvo em ../data/processed/online_retail_clean.csv")
print(f"Tamanho final: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

Dataset salvo em ../data/processed/online_retail_clean.csv
Tamanho final: 770,715 linhas x 9 colunas
